# Batch ATS superframe processing and visualization

This notebook imports all `.ats` files in one folder and applies the same style of processing/visualization as `flir_ats_superframing_clean_v2`, but **only on reconstructed superframes**.

Important differences from the older notebook:

- no raw preset 0 / preset 1 processing;
- no manual HDR-like merge;
- no exported `.npz` files;
- all statistics and plots are computed from `read_ats_file(..., read_mode="superframe")`.


## 1. User settings

Edit these paths before running the notebook.


In [ ]:
from pathlib import Path

# Folder containing your .ats files
# ATS_FOLDER = Path('/Volumes/Extreme SSD/GlobalAM-Dataset FINAL/ATS build 1 - without presintering')
# ATS_FOLDER = Path('/Volumes/Extreme SSD/GlobalAM-Dataset FINAL/ATS build 2 - with presintering')
ATS_FOLDER = Path('/Users/matte/Documents/ATS build 1 - without presintering')

# Folder containing ats_reader.py and temperature_exploration.py
# Use Path.cwd() if these files are in the same folder as this notebook.
CODE_DIR = Path.cwd()

# Optional: process only the first N files while testing. Use None for all files.
MAX_FILES = None

# Reusable object-parameter config passed to the ATS reader
OBJECT_PARAMETERS_CONFIG = CODE_DIR / 'object_parameters.json'

# Optional maximum number of superframes to read per video. Use None for all.
MAX_FRAMES_PER_VIDEO = None

# Optional selected frame for image visualization below; 1 means first superframe.
SELECTED_SUPERFRAME_NUMBER = 100

# Optional crop coordinates for crop visualization: frames are indexed as [y, x]
Y_MIN, Y_MAX = 300, 400
X_MIN, X_MAX = 150, 250


## 2. Imports

This cell imports your local ATS reader utilities. If it fails, check `CODE_DIR` and confirm that it contains `ats_reader.py`.


In [ ]:
import sys
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CODE_DIR = CODE_DIR.expanduser().resolve()
OBJECT_PARAMETERS_CONFIG = OBJECT_PARAMETERS_CONFIG.expanduser().resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from ats_reader import (
    inspect_ats_file,
    list_ats_files,
    load_object_parameter_updates,
    read_ats_file,
)

print(f'Imported ATS utilities from: {CODE_DIR}')
print(f'ATS folder: {ATS_FOLDER}')
print(f'Object-parameter config: {OBJECT_PARAMETERS_CONFIG}')


## 3. Find `.ats` files

In [ ]:
ATS_FOLDER = ATS_FOLDER.expanduser().resolve()

ats_files = list_ats_files(ATS_FOLDER, max_files=MAX_FILES)

print(f'Found {len(ats_files)} ATS files')
for f in ats_files:
    print(' -', f.name)

if not ats_files:
    raise FileNotFoundError(f'No .ats files found in {ATS_FOLDER}')


## 4. Helper functions

`list_ats_files()`, `inspect_ats_file()`, and `read_ats_file()` are imported from `ats_reader.py` because they are part of the data import path. The helpers below stay in the notebook because they are analysis/presentation choices for this workflow.


In [ ]:
def get_layer_number(name):
    """Extract layer number from filenames containing e.g. 'layer40'."""
    match = re.search(r'layer(\d+)', name, flags=re.IGNORECASE)
    return int(match.group(1)) if match else float('inf')


def compute_superframe_statistics(frames: np.ndarray) -> pd.DataFrame:
    """Compute frame-wise statistics from a superframe stack shaped (n_frames, height, width)."""
    flat = frames.reshape(frames.shape[0], -1)

    stats = pd.DataFrame({
        'superframe_id': np.arange(1, frames.shape[0] + 1),
        'min': np.nanmin(flat, axis=1),
        'max': np.nanmax(flat, axis=1),
        'mean': np.nanmean(flat, axis=1),
        'std': np.nanstd(flat, axis=1),
        'p05': np.nanpercentile(flat, 5, axis=1),
        'p95': np.nanpercentile(flat, 95, axis=1),
    })
    stats['range'] = stats['max'] - stats['min']
    return stats


## 5. Read all videos as superframes and compute statistics

This creates:

- `all_results`: dictionary with one entry per ATS video;
- `summary_table`: compact table with one row per video.


In [ ]:
all_results = {}
summary_rows = []
object_parameters = load_object_parameter_updates(OBJECT_PARAMETERS_CONFIG)
print('Object parameters applied before reading:')
if object_parameters:
    for parameter_name, parameter_value in sorted(object_parameters.items()):
        print(f'  {parameter_name}: {parameter_value}')
else:
    print('  none')

for ats_path in ats_files:
    print(f'Processing: {ats_path.name}')

    inspection = inspect_ats_file(
        str(ats_path),
        object_parameter_updates=object_parameters,
        collect_preset_value_ranges=False,
    )
    if not inspection.is_superframe:
        raise ValueError(f'{ats_path.name} is not reported as a superframing ATS file.')

    ats_data = read_ats_file(
        str(ats_path),
        read_mode='superframe',
        object_parameter_updates=object_parameters,
        inspection=inspection,
        max_frames=MAX_FRAMES_PER_VIDEO,
    )

    frames = ats_data.frames
    stats = compute_superframe_statistics(frames)

    video_name = ats_path.stem
    all_results[video_name] = {
        'path': ats_path,
        'ats_data': ats_data,
        'frames': frames,
        'stats': stats,
    }

    summary_rows.append({
        'video': video_name,
        'n_superframes': frames.shape[0],
        'height': frames.shape[1],
        'width': frames.shape[2],
        'global_min': np.nanmin(frames),
        'global_mean': np.nanmean(frames),
        'global_max': np.nanmax(frames),
        'mean_of_frame_means': stats['mean'].mean(),
        'mean_of_frame_stds': stats['std'].mean(),
    })

summary_table = pd.DataFrame(summary_rows).sort_values(
    by='video',
    key=lambda s: s.map(get_layer_number),
).reset_index(drop=True)

print('Done.')
display(summary_table)


## 6. Three-panel plot: min, max, and mean intensity of superframes

Each curve corresponds to one `.ats` video. All values are computed on reconstructed superframes only.


In [ ]:
sorted_items = sorted(
    all_results.items(),
    key=lambda item: get_layer_number(item[0])
)

fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

plot_specs = [
    ('min', 'Minimum intensity'),
    ('max', 'Maximum intensity'),
    ('mean', 'Mean intensity'),
]

for ax, (column, title) in zip(axes, plot_specs):
    for video_name, results in sorted_items:
        stats = results['stats']
        layer_id = get_layer_number(video_name)
        label = f'layer_{layer_id}' if np.isfinite(layer_id) else video_name
        ax.plot(stats['superframe_id'], stats[column], label=label, linewidth=0.8)

    ax.set_title(f'Superframe - {title}')
    ax.set_ylabel(title)
    ax.grid(True)

# Set only for the third panel (mean).
axes[2].set_ylim(306, 312)

axes[-1].set_xlabel('Superframe order')
axes[0].legend(fontsize=8, loc='best', ncol=2)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from pathlib import Path

# Your target folder
CSV_OUTPUT_DIR = Path.cwd() / "intensity_statistics_csv"

# Create folder if it doesn't exist
CSV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for video_name, results in sorted_items:
    stats = results["stats"]

    df = pd.DataFrame({
        "superframe_id": stats["superframe_id"],
        "min_intensity": stats["min"],
        "max_intensity": stats["max"],
        "mean_intensity": stats["mean"],
    })

    # Clean filename (remove problematic characters)
    safe_name = Path(video_name).stem.replace(" ", "_")

    output_path = CSV_OUTPUT_DIR / f"{safe_name}_intensity_statistics.csv"

    df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")

## 7. Additional plots transferred from the old notebook, but on superframes

The old notebook plotted standard deviation and mean separately for low/high subframes. Here there is only one merged superframe sequence, so the analogous plot is `std` and `mean` of the reconstructed superframes.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

for video_name, results in sorted_items:
    stats = results['stats']
    layer_id = get_layer_number(video_name)
    label = f'layer_{layer_id}' if np.isfinite(layer_id) else video_name

    axes[0].plot(stats['superframe_id'], stats['std'], label=label, linewidth=0.8)
    axes[1].plot(stats['superframe_id'], stats['mean'], label=label, linewidth=0.8)

axes[0].set_title('Superframe - standard deviation of intensity')
axes[1].set_title('Superframe - mean intensity')

axes[0].set_ylabel('Standard deviation')
axes[1].set_ylabel('Mean')
axes[1].set_xlabel('Superframe order')

for ax in axes:
    ax.grid(True)

axes[0].legend(fontsize=8, loc='best', ncol=2)

plt.tight_layout()
plt.show()


## 8. Mean with 5-95 percentile band for one selected video

This reproduces the mean + percentile-band logic, but using the merged superframe image at each time step.


In [ ]:
# Select the first video by default. Change SELECTED_VIDEO_NAME to a key in all_results if needed.
SELECTED_VIDEO_NAME = sorted_items[0][0]

stats = all_results[SELECTED_VIDEO_NAME]['stats']

plt.figure(figsize=(10, 5))
plt.plot(stats['superframe_id'], stats['mean'], linewidth=1.5, label='Mean')
plt.fill_between(
    stats['superframe_id'],
    stats['p05'],
    stats['p95'],
    alpha=0.3,
    label='5-95 percentile',
)

plt.xlabel('Superframe order')
plt.ylabel('Intensity')
plt.title(f'Superframe intensity over time - {SELECTED_VIDEO_NAME}')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 9. Visualize one reconstructed superframe

This replaces the old side-by-side raw subframe 0 / raw subframe 1 / manual merge plot. Here the displayed frame is directly the reconstructed superframe returned by the reader.


In [ ]:
SELECTED_VIDEO_NAME = sorted_items[1][0] # LAYER 40
frames = all_results[SELECTED_VIDEO_NAME]['frames']

SELECTED_SUPERFRAME_NUMBERS = [689, 762, 837, 913, 988, 1064, 1139, 1215]  # WITHOUT PRESINTERING
#SELECTED_SUPERFRAME_NUMBERS = [779, 826, 949, 1034, 1119, 1204, 1289, 1375] #WITH PRESINTERING

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for ax, superframe_number in zip(axes, SELECTED_SUPERFRAME_NUMBERS):

    frame_index = superframe_number - 1

    if frame_index < 0 or frame_index >= frames.shape[0]:
        raise IndexError(
            f'SELECTED_SUPERFRAME_NUMBER={superframe_number} is outside '
            f'the available range 1..{frames.shape[0]} for {SELECTED_VIDEO_NAME}'
        )

    frame = frames[frame_index]

    im = ax.imshow(frame, cmap='inferno', vmin = 310, vmax=400)
    ax.set_title(f'Superframe {superframe_number}', fontsize=10)
    ax.axis('off')

# Increase vertical spacing.
fig.subplots_adjust(hspace=0.08, wspace=0.01)

# Leave room on the right for colorbar.
fig.subplots_adjust(right=0.90)

# Create a dedicated axis for colorbar outside the panels.
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Intensity')

fig.suptitle(f'{SELECTED_VIDEO_NAME} - reconstructed superframes', fontsize=14)

plt.show()


## 10. Violinplots of layer 40 within the ROIs (from binary mask)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from scipy import ndimage

# -----------------------------
# Inputs
# -----------------------------
SELECTED_VIDEO_NAME = sorted_items[1][0]  # LAYER 40
frames = all_results[SELECTED_VIDEO_NAME]["frames"]   # shape: (n_superframes, rows, cols)

MASK_PATH = Path('/Volumes/Extreme SSD/GlobalAM-Dataset FINAL/Mask_build 1 - without_presintering.png')  # WITHOUT PRESINTERING
#MASK_PATH = Path('/Volumes/Extreme SSD/GlobalAM-Dataset FINAL/Mask_build 2 - with_presintering.png')  # WITH PRESINTERING

# Rows = [start_superframe, end_superframe], 1-based and inclusive     #WITHOUT PRESINTERING
X = np.array([
    [21, 663],
    [686, 1290],
    [1311, 1778],
    [1798, 2170],
    [2194, 2428],
    [2461, 2682],
    [2706, 2879],
    [2893, 3025],
])

# Rows = [start_superframe, end_superframe], 1-based and inclusive     #WITH PRESINTERING
# X = np.array([
#     [41, 716],
#     [779, 1460],
#     [1526, 2210],
#     [2254, 2927],
#     [2966, 3232],
#     [3282, 3550],
#     [3661, 3936],
#     [4023, 4294],
# ])

# -----------------------------
# Load grayscale mask
# -----------------------------
mask_img = Image.open(MASK_PATH).convert("L")
mask_gray = np.array(mask_img).astype(float)

# Normalize to 0-1 if image is 0-255
if mask_gray.max() > 1:
    mask_gray = mask_gray / 255.0

# Background is white ~1; foreground components are darker
foreground = mask_gray < 0.99

# Optional sanity check
plt.figure(figsize=(5, 5))
plt.imshow(foreground, cmap="gray")
plt.title("Detected foreground components")
plt.axis("off")
plt.show()

# -----------------------------
# Label connected components
# -----------------------------
labeled_mask, n_components = ndimage.label(foreground)

print(f"Detected connected components: {n_components}")

if n_components < X.shape[0]:
    raise ValueError(
        f"Detected only {n_components} connected components, "
        f"but X contains {X.shape[0]} frame ranges."
    )

if labeled_mask.shape != frames.shape[1:]:
    raise ValueError(
        f"Mask shape {labeled_mask.shape} does not match frame shape {frames.shape[1:]}."
    )

# -----------------------------
# Order components by mask grayscale intensity
# lowest intensity = first row of X
# highest foreground intensity = last row of X
# -----------------------------
component_info = []

for comp_id in range(1, n_components + 1):
    comp_pixels = labeled_mask == comp_id
    mean_gray = np.mean(mask_gray[comp_pixels])
    component_info.append((comp_id, mean_gray))

component_info_sorted = sorted(component_info, key=lambda x: x[1])
ordered_component_ids = [comp_id for comp_id, _ in component_info_sorted]

print("\nComponent order based on mask grayscale intensity:")
for order, (comp_id, mean_gray) in enumerate(component_info_sorted, start=1):
    print(f"Order {order}: component {comp_id}, mean mask intensity = {mean_gray:.4f}")

# Use only as many components as rows in X
ordered_component_ids = ordered_component_ids[:X.shape[0]]

# -----------------------------
# Extract mean intensity per superframe inside each component
# -----------------------------
boxplot_data = []
boxplot_labels = []

for row_id, (start_sf, end_sf) in enumerate(X, start=1):

    comp_id = ordered_component_ids[row_id - 1]

    # Convert 1-based inclusive superframe numbers to Python slicing
    start_idx = start_sf - 1
    end_idx = end_sf

    if start_idx < 0 or end_idx > frames.shape[0]:
        raise IndexError(
            f"Range {start_sf}-{end_sf} is outside available superframes "
            f"1..{frames.shape[0]}"
        )

    frame_stack = frames[start_idx:end_idx]
    comp_pixels = labeled_mask == comp_id

    mean_intensity_per_frame = np.nanmean(frame_stack[:, comp_pixels], axis=1)

    boxplot_data.append(mean_intensity_per_frame)
    boxplot_labels.append(f"CC {comp_id}\nSF {start_sf}-{end_sf}")
    custom_labels = [
    "Sub.1 Part 1", "Sub.1 Part 2", "Sub.1 Part 3", "Sub.1 Part 4",
    "Sub.2 Part 1", "Sub.2 Part 2", "Sub.2 Part 3", "Sub.2 Part 4"
    ]

    print(
        f"X row {row_id}: superframes {start_sf}-{end_sf} "
        f"-> component {comp_id}, "
        f"n={len(mean_intensity_per_frame)} frames"
    )

# -----------------------------
# Violin plot
# -----------------------------
plt.figure(figsize=(11, 5))

# Violin plot
parts = plt.violinplot(
    boxplot_data,
    showmeans=False,
    showmedians=True,
    showextrema=False
)

# Optional styling
for pc in parts['bodies']:
    pc.set_facecolor('#1f77b4')
    pc.set_alpha(0.4)

# Overlay points (jittered)
for i, data in enumerate(boxplot_data, start=1):
    x = np.random.normal(i, 0.04, size=len(data))  # small horizontal jitter
    plt.scatter(x, data, s=8, alpha=0.4, color='k')

# X ticks
plt.xticks(
    ticks=np.arange(1, len(boxplot_labels) + 1),
    labels=custom_labels
)

plt.xlabel("Sample")
plt.ylim(301, 493)
plt.ylabel("Mean intensity within ROI")
plt.title("Mean superframe intensity during laser exposure")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()